# Fruit Classification with VGG16
**Author: Ibrahim Al-Hashimi**

An educational transfer-learning pipeline based on the supplied notebook, refactored into reusable modules. This notebook shares code with the command-line workflow. No cells have been executed in the delivered package; outputs below are generated only when you run it.

## Project overview
Train a custom classification head on a frozen ImageNet VGG16 backbone, fine-tune block5, select the checkpoint with the best full-validation loss, then evaluate once on the full test split. See `README.md` and `README_AR.md` for setup and limitations.


## 1. Environment
Use Python 3.12. Install `requirements-notebook.txt` from the project root before opening this notebook. Restart the kernel after changing packages. The default run uses pretrained ImageNet weights and needs an internet connection on first use.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.utils import load_config, inspect_dataset, read_json
config = load_config(ROOT / "config.json")
print("Project:", ROOT)
print("Input size:", config["image_size"], "Batch size:", config["batch_size"])


## 2. About the dataset
The source notebook identifies the archive as Fruits-360 original-size and supplies an IBM Skills Network mirror. Images are organized into Training, Validation and Test folders with class subdirectories. Exact counts are computed below rather than assumed. The supplied splits are preserved; object-level leakage has not been independently audited. Review `data/README.md` and the exact archive's license before redistributing images.


In [ ]:
from src.download_data import download_data
summary = download_data(config)
base, class_names, summary = inspect_dataset(config["data_dir"])
print("Classes:", summary["num_classes"])
print("Split totals:", summary["totals"])
print("First classes:", class_names[:10])


In [ ]:
import matplotlib.pyplot as plt
counts = summary["counts_per_class"]["Training"]
fig, ax = plt.subplots(figsize=(max(10, len(counts) * 0.25), 4))
ax.bar(list(counts), list(counts.values()))
ax.set(title="Training images per class", xlabel="Class", ylabel="Images")
ax.tick_params(axis="x", labelrotation=90)
plt.tight_layout()
plt.show()


## 3. Preprocessing and architecture
`src/data.py` applies training-only augmentation and VGG16 `preprocess_input`. Inputs are **not divided by 255**. Validation/test loaders do not augment or shuffle. Image resizing uses bilinear interpolation everywhere.

The architecture in `src/model.py` is VGG16 → GlobalAveragePooling → Dense(256, ReLU) → BatchNormalization → Dropout(0.3) → Softmax. The output width equals the discovered class count. Input resolution is 64 × 64 to match the starting experiment.

## 4. Train and fine-tune
Training uses all batches in each split, seeded operations, separate callbacks per stage, and checkpoints selected by full-validation loss. A nonempty output folder is protected from overwriting. Change output_dir/reports_dir in config.json before another run. Full training may take substantial time on CPU.


In [ ]:
from src.train import train
history = train(config)


## 5. Inspect learning curves
The fine-tuning phase continues on the epoch axis after the head-training phase. These curves diagnose learning behavior; they do not replace evaluation on independent test images.


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(Path(config["reports_dir"]) / "figures" / "training_curves.png")))
print(read_json(Path(config["output_dir"]) / "selection.json"))


## 6. Full test-set evaluation
The saved best model is evaluated on every test image. This exports accuracy, macro/weighted F1, cross-entropy, per-class metrics, a confusion matrix, and all predictions. Do not use test results to choose the checkpoint or tune repeated experiments.


In [ ]:
from src.evaluate import evaluate
metrics = evaluate(config)
metrics


In [ ]:
display(Image(filename=str(Path(config["reports_dir"]) / "figures" / "confusion_matrix.png")))
display(Image(filename=str(Path(config["reports_dir"]) / "figures" / "sample_predictions.png")))


## 7. Predict a single image
The example chooses one actual test image. Replace `image_path` with your own image to explore domain shift. Labels and preprocessing are loaded from the saved model bundle. Softmax scores are not calibrated confidence or unknown-class detection.


In [ ]:
from src.predict import predict_image
from src.utils import IMAGE_EXTENSIONS
image_path = next(p for p in (base / "Test").rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
result = predict_image(image_path, config["output_dir"], top_k=3)
display(Image(filename=str(image_path), width=240))
result


## 8. Report and limitations
Copy only measured results into README. Keep dataset version/hash, configuration and environment with the model. Uniform-background dataset performance does not imply reliability on real-world photos. Similar views can be correlated, and this project has not established object-level independence across splits.

References: [Keras VGG16](https://keras.io/api/applications/vgg/), [Keras transfer learning](https://keras.io/guides/transfer_learning/), and `THIRD_PARTY_NOTICES.md` for source-notebook and dataset attribution.
